# Phase 2: Feature Engineering
Machine Learning models don't understand raw dates like `2021-10-01 06:39:49`. They need numbers. 
In this phase, we will extract useful features out of the raw timestamps and spatial IDs.

In [1]:
import pandas as pd
import numpy as np

# 1. Load the data and filter outliers again (as we did in Phase 1)
df = pd.read_csv('../data/raw/archive/bus_running_times_654.csv')
df = df[(df['run_time_in_seconds'] > 0) & (df['run_time_in_seconds'] <= 1800)].copy()
print(f"Loaded {len(df)} clean rows.")

Loaded 200679 clean rows.


In [4]:
# 2. Parse Datetime
# Combine date and start_time into a single datetime column
df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['start_time'])

# Extract temporal features
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

# Rush hour indicator (e.g., 7-9 AM and 4-7 PM)
df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)

display(df[['datetime', 'hour', 'day_of_week', 'is_rush_hour']].head())

,datetime,hour,day_of_week,is_rush_hour
0,2021-10-01 06:39:49,6,4,0
1,2021-10-01 06:42:12,6,4,0
2,2021-10-01 06:45:42,6,4,0
3,2021-10-01 06:54:04,6,4,0
4,2021-10-01 06:57:19,6,4,0


In [5]:
# 3. Cyclical Encoding for Time (Crucial ML concept!)
# Hour 23 and Hour 0 are 1 hour apart, but numerically 23 units apart.
# We use sin/cos transforms so the ML model understands they are close to each other.
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 23.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 23.0)

In [6]:
# 4. Historical Segment Averages (Target Encoding)
# How long does this specific segment USUALLY take at this hour?
segment_hourly_avg = df.groupby(['segment', 'hour'])['run_time_in_seconds'].mean().reset_index()
segment_hourly_avg.rename(columns={'run_time_in_seconds': 'avg_segment_time_at_hour'}, inplace=True)

# Merge it back into our main dataframe
df = pd.merge(df, segment_hourly_avg, on=['segment', 'hour'], how='left')

# Fill any missing averages with the global segment average
segment_global_avg = df.groupby('segment')['run_time_in_seconds'].mean()
df['avg_segment_time_at_hour'] = df['avg_segment_time_at_hour'].fillna(df['segment'].map(segment_global_avg))

In [8]:
# 5. Select final features and save to processed folder
features = [
    'segment', 'direction', 'length', 
    'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 
    'hour_sin', 'hour_cos', 'avg_segment_time_at_hour', 
    'run_time_in_seconds' # Target variable
]

processed_df = df[features].copy()

processed_file_path = '../data/processed/engineered_running_times.csv'
processed_df.to_csv(processed_file_path, index=False)
print(f"Successfully saved processed dataset to {processed_file_path}!")
display(processed_df.head())

Successfully saved processed dataset to ../data/processed/engineered_running_times.csv!


,segment,direction,length,hour,day_of_week,is_weekend,is_rush_hour,hour_sin,hour_cos,avg_segment_time_at_hour,run_time_in_seconds
0,1.0,1.0,0.6261,6,4,0,0,0.997669,-0.068242,97.517560,69.0
1,2.0,1.0,1.2808,6,4,0,0,0.997669,-0.068242,251.755511,210.0
2,3.0,1.0,2.1125,6,4,0,0,0.997669,-0.068242,474.046875,496.0
3,4.0,1.0,1.5513,6,4,0,0,0.997669,-0.068242,200.345679,195.0
4,5.0,1.0,0.8450,6,4,0,0,0.997669,-0.068242,117.498233,97.0
